# 🚀 Тестирование Agent Framework

Этот ноутбук пошагово проверяет все компоненты фреймворка агентов:

1. **Тест базовых функций** - проверка инструментов
2. **Test Agent** - простейший агент (1 состояние)
3. **My Agent** - переходы между состояниями (2 состояния)
4. **Router Agent** - условный роутинг (ветвление)
5. **Multi-Agent System** - один агент вызывает другого
6. **Audit Agent** - полный тест (6 состояний)

Бэкенд (GigaChat или LM Studio) настраивается в `config.yaml`.

In [1]:
# Установка зависимостей (запустите один раз)
# %pip install -r requirements.txt

## 📦 Общая настройка

Импорты и конфигурация для всех тестов.

In [1]:
# Импорты
import yaml

# Импорты агентов
from src.agents.test_agent import TestAgent
from src.agents.my_agent import MyAgent
from src.agents.router_agent import RouterAgent
from src.agents.supervisor_agent import SupervisorAgent
from src.agents.audit_agent import AuditAgent

# Импорты инструментов и клиентов
from src.tools.tools import (
    get_tools_dict,
    reset_memory,
    register_agent,
    calculator,
)
from src.connections.clients import get_llm_client

# Logging (настройка уровня в config.yaml -> logging.level)
from src.agent_engine import init_logging

print("✓ Импорты выполнены успешно")

# Загрузка конфигурации
with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

backend = config['active_backend']
recursion_limit = config['agent']['recursion_limit']

print(f"Активный бэкенд: {backend}")
print(f"Лимит рекурсии графа: {recursion_limit}")

# Создание LLM клиента
llm = get_llm_client(backend, config)
print(f"✓ LLM клиент создан: {type(llm).__name__}")

# Инициализация logging (уровень задаётся в config.yaml)
init_logging()
print("✓ Logging инициализирован")

✓ Импорты выполнены успешно
Активный бэкенд: lmstudio
Лимит рекурсии графа: 30
✓ LLM клиент создан: ChatOpenAI
logging: level=detailed
✓ Logging инициализирован


---

## 1️⃣ Тест базовых функций

Проверяем что инструменты работают корректно.

In [6]:
# print("=" * 60)
# print("ТЕСТ 1: Базовые функции")
# print("=" * 60)

# # Тест калькулятора
# print("\n1. Тест калькулятора:")
# result = calculator.invoke("2 ** 10")
# print(f"   2^10 = {result}")
# assert result == "1024", "Калькулятор работает неправильно!"
# print("   ✓ Калькулятор работает")

# # Тест памяти
# print("\n2. Тест памяти:")
# from src.tools.tools import memory
# reset_memory()
# memory.invoke({"action": "save", "key": "test_key", "value": "test_value"})
# result = memory.invoke({"action": "get", "key": "test_key"})
# assert "test_value" in result, "Память работает неправильно!"
# print("   ✓ Память работает")

# print("\n✅ Все базовые функции работают корректно!")

In [8]:
# print("=" * 60)
# print("ТЕСТ 1.5: Проверка подключения к LLM")
# print("=" * 60)

# from langchain_core.messages import HumanMessage

# try:
#     test_response = llm.invoke([HumanMessage(content="Ответь одним словом: 2+2=")])
#     print(f"\n✓ LLM ответил: {test_response.content.strip()}")
#     print("✓ Подключение к LLM работает")
# except Exception as e:
#     print(f"\n✗ Ошибка подключения к LLM: {e}")
#     print("Проверьте что LM Studio запущен и доступен на порту из config.yaml")

---

## 2️⃣ Test Agent - Простейший агент

Граф: `[work] → END`

Проверяем:
- Создание агента через AgentConfig
- Работу с инструментами
- Переход в END по ключевому слову

In [7]:
print("======== ТЕСТ 2: Test Agent (1 состояние) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="test_agent")

# Создание агента
test_agent = TestAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {test_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(test_agent.visualize())

======== ТЕСТ 2: Test Agent (1 состояние) ========

✓ Агент создан: TestAgent(id=TestAgent_1305907637824, states=1)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: work
  - Состояний: 1
  - Уникальных тулов: 3
  - Ключей в памяти сейчас: 0

Состояния:
  - work (entry)
    Описание: Единственное рабочее состояние
    Переходы: END
    Инструменты (shared): calculator, memory, think
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [ ]:
result = test_agent.invoke([
    "Посчитай 15 * 23 и сохрани результат в память с ключом 'result'"
])

RUN a6672c31 | TestAgent_1306702618048 | started
STATE START -> work
  REQ 1 | msgs=2 in=607 out=37
    [SYS] Ты простой тестовый агент.

Твои возможности:
- calculator: вычисление математических выражений
- memory: сохранение/чтение данных
- think: размышления

Выполни запрос пользователя, используя доступные инструменты.


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вызови инструмент transition:
- reasoning: почему переходишь именно туда.
- summary: что было сделано и что нужно сделать в следующем состоянии. Укажи какие ключи сохранены в memory.
- next_state: одно из доступных значений:
  - "stay" — остаться в текущем состоянии (нужно ещё поработать)
  - "END"

ВАЖНО: после вызова transition не вызывай другие инструменты — напиши финальный ответ.
    [USER] Посчитай 15 * 23 и сохрани результат в память с ключом 'result'
  TOOL calculator params={'expression': '15 * 23'}
  TOOL memory_append params= 15 * 23 = 345
    -> ✓ Добавлено в журнал:  15 * 23 = 34

---

## 3️⃣ My Agent - Переходы между состояниями

Граф: `[work] ⟳ → [summarize] → END`

Проверяем:
- Циклическое состояние (work возвращается в себя)
- Условный переход по ключевому слову
- Работу с памятью между состояниями

In [8]:
print("======== ТЕСТ 3: My Agent (2 состояния + переход) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="my_agent")

# Создание агента
my_agent = MyAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {my_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(my_agent.visualize())

======== ТЕСТ 3: My Agent (2 состояния + переход) ========

✓ Агент создан: MyAgent(id=MyAgent_1305907635568, states=2)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: work
  - Состояний: 2
  - Уникальных тулов: 5
  - Ключей в памяти сейчас: 0

Состояния:
  - work (entry)
    Описание: Основное рабочее состояние с полным набором инструментов
    Переходы: summarize
    Инструменты (shared): calculator, ask_human, think, memory
    Memory injections: нет
  - summarize
    Описание: Финальное состояние для рефлексии и подведения итогов
    Переходы: END
    Инструменты (shared): summarize, memory
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [9]:
# Запуск агента
result = my_agent.invoke([
    "давай подсчитаем площадь круга?"
])

# Вывод результата
print("\n✅ My Agent работает корректно!")

RUN 710a325b | MyAgent_1305907635568 | started
STATE START -> work
  REQ 1 | msgs=2 in=735 out=37
    [SYS] Ты полезный помощник для математических вычислений.

Твои возможности:
- calculator: вычисление математических выражений
- ask_human: задать вопрос пользователю, если не хватает данных
- think: зафиксировать свои размышления
- memory: сохранить промежуточные результаты или прочитать сохраненное

Алгоритм работы:
1. Проанализируй запрос пользователя
2. Если не хватает данных - используй ask_human для уточнения
3. Используй think для размышлений о подходе к решению
4. Выполни вычисления через calculator
5. Сохрани важные результаты через memory


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вызови инструмент transition:
- reasoning: почему переходишь именно туда.
- summary: что было сделано и что нужно сделать в следующем состоянии. Укажи какие ключи сохранены в memory.
- next_state: одно из доступных значений:
  - "stay" — остаться в текущем состоянии 

---

## 4️⃣ Router Agent - Условный роутинг

Граф: `[classify] → [math | text | error] → END`

Проверяем:
- Классификацию запроса
- Роутер с несколькими выходами
- Разные пути обработки

In [ ]:
print("======== ТЕСТ 4: Router Agent (условный роутинг) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="router_agent")

# Создание агента
router_agent = RouterAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {router_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(router_agent.visualize())

ТЕСТ 4: Router Agent (условный роутинг)

✓ Агент создан: RouterAgent(id=RouterAgent_2143587284992, states=4)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: classify
  - Состояний: 4
  - Уникальных тулов: 3
  - Ключей в памяти сейчас: 0

Состояния:
  - classify (entry)
    Описание: Классификация типа запроса
    Переходы: math, text, error
    Инструменты: think [shared], memory [shared]
    Memory injections: request_type
  - math
    Описание: Обработка математических запросов
    Переходы: END
    Инструменты: calculator [shared], memory [shared], think [shared]
    Memory injections: request_type, result
  - text
    Описание: Обработка текстовых запросов
    Переходы: END
    Инструменты: memory [shared], think [shared]
    Memory injections: request_type, response_text
  - error
    Описание: Обработка нераспознанных запросов
    Переходы: END
    Инструменты: think [shared]
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найден

In [ ]:
# Тест 1: Математический запрос
print("======== Тест 4.1: Математический запрос ========")

reset_memory()
result = router_agent.invoke(["Посчитай 5 + 5"])


Тест 4.1: Математический запрос



============================================================

RUN 05a6ed36 | RouterAgent_1670092436400 | started

============================================================

STATE START -> classify

...

result: 10

============================================================

RUN 05a6ed36 | RouterAgent_1670092436400 | 26.8s | 5342 tokens (5029 in / 313 out) | 7 LLM calls, 6 tool calls

============================================================


Результат:
Результат вычисления 5 + 5 равен **10**.

✓ Математический путь работает


In [ ]:
# Тест 2: Текстовый запрос
print("======== Тест 4.2: Текстовый запрос ========")

reset_memory()
result = router_agent.invoke(["Привет! Как дела?"])


Тест 4.2: Текстовый запрос



============================================================

RUN 4ffdf669 | RouterAgent_1670092436400 | started

============================================================

STATE math -> classify

...

request_type: text

============================================================

RUN 4ffdf669 | RouterAgent_1670092436400 | 27.8s | 5216 tokens (4921 in / 295 out) | 7 LLM calls, 5 tool calls

============================================================


Результат:
Привет! У меня всё отлично, спасибо за вопрос. Как у тебя дела?

✓ Текстовый путь работает

✅ Router Agent работает корректно!


---

## 5️⃣ Multi-Agent System - Композиция агентов

Граф: `Supervisor [delegate] → [aggregate] → END`
         ↓ вызывает ↓
       Test Agent, Router Agent

Проверяем:
- Регистрацию агентов в реестре
- Вызов агента из агента через инструмент call_agent
- Агрегацию результатов от нескольких агентов

In [ ]:
print("======== ТЕСТ 5: Multi-Agent System (композиция агентов) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="supervisor_agent")

# Создаем подчиненных агентов
test_agent_sub = TestAgent(llm, tools_dict, agent_id="test_agent_sub")
router_agent_sub = RouterAgent(llm, tools_dict, agent_id="router_agent_sub")

print("\n✓ Подчиненные агенты созданы:")
print(f"  - {test_agent_sub}")
print(f"  - {router_agent_sub}")

# Регистрируем их в реестре
register_agent("test_agent", test_agent_sub)
register_agent("router_agent", router_agent_sub)

print("\n✓ Агенты зарегистрированы в реестре")

ТЕСТ 5: Multi-Agent System (композиция агентов)

✓ Подчиненные агенты созданы:
  - TestAgent(id=test_agent_sub, states=1)
  - RouterAgent(id=router_agent_sub, states=4)

✓ Агенты зарегистрированы в реестре


In [4]:
# Создаем супервизора с инструментом call_agent
supervisor_tools_dict = get_tools_dict(agent_name="supervisor_agent")
supervisor = SupervisorAgent(llm, supervisor_tools_dict)

print(f"✓ Супервизор создан: {supervisor}")

# Визуализация графа
print("\nСтруктура графа супервизора:")
print(supervisor.visualize())

✓ Супервизор создан: SupervisorAgent(id=SupervisorAgent_2144945434272, states=2)

Структура графа супервизора:
Граф агента (preflight):

Сводка:
  - Entry point: delegate
  - Состояний: 2
  - Уникальных тулов: 4
  - Ключей в памяти сейчас: 0

Состояния:
  - delegate (entry)
    Описание: Делегирование задач специализированным агентам
    Переходы: aggregate
    Инструменты: call_agent [shared], memory [shared], think [shared]
    Memory injections: request_type, response_text, delegation_notes
  - aggregate
    Описание: Агрегация результатов от агентов
    Переходы: END
    Инструменты: memory [shared], summarize [shared], think [shared]
    Memory injections: request_type, response_text, delegation_notes

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [7]:
# Запуск супервизора
print("\n" + "=" * 60)
print("Запуск: Посчитай 10 * 5 и поздоровайся")
print("=" * 60 + "\n")

result = supervisor.invoke([
    "Посчитай 10 * 5 через test_agent и передай приветствие через router_agent"
])

# Вывод результата
print("\n" + "=" * 60)
print("Результат:")
print("=" * 60)
print(result['messages'][-1].content)

print("\n✅ Multi-Agent System работает корректно!")


Запуск: Посчитай 10 * 5 и поздоровайся



============================================================

RUN 0e5df44c | SupervisorAgent_1670096965696 | started

============================================================

STATE START -> delegate

...

request_type: text

============================================================

RUN None | SupervisorAgent_1670096965696 | 99.6s | 16157 tokens (14882 in / 1275 out) | 17 LLM calls, 9 tool calls

============================================================


Результат:
**Результат выполнения**

- **Умножение:** \(10 \times 5 = 50\)  
- **Приветствие от `router_agent`:** «Hello!»

Если понадобится продолжить работу или задать новый вопрос — дайте знать!

✅ Multi-Agent System работает корректно!


---

## 6️⃣ Audit Agent - Полный тест

Граф: `[start_work] → [analize_word] → [analize_sql] → [analize_py] → [self_check] ⟳ → [write_report] → END`

Проверяем:
- Сложный многошаговый workflow
- Роутер с самопроверкой
- Работу с файловой системой
- Анализ разных типов файлов
- Пакетный **memory** в `start_work`: после `list_case_files` шесть ключей (`case_id` … `py_files`) сохраняются **одним** вызовом `memory(action="save", keys=[...], values=[...])` (см. промпт `AuditAgent` и запрос в ячейке ниже)

In [2]:
print("======== ТЕСТ 6: Audit Agent (полный тест) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="audit_agent")

# Создание агента
audit_agent = AuditAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {audit_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(audit_agent.visualize())

======== ТЕСТ 6: Audit Agent (полный тест) ========

✓ Агент создан: AuditAgent(id=AuditAgent_2055487539440, states=5)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: start_work
  - Состояний: 5
  - Уникальных тулов: 9
  - Ключей в памяти сейчас: 0

Состояния:
  - start_work (entry)
    Описание: Поиск папки проверки и сбор списка файлов
    Переходы: analize_sql
    Инструменты (shared): ask_human, think, memory
    Инструменты (local): list_data_folders, find_case_folder, list_case_files
    Memory injections: case_id, case_folder, case_files, docx_files, sql_files, py_files
  - analize_sql
    Описание: Анализ SQL скриптов на риски и проблемы
    Переходы: analize_py, start_work
    Инструменты (shared): memory, think
    Инструменты (local): read_sql_file
    Memory injections: case_files, sql_files, sql_verdicts
  - analize_py
    Описание: Анализ Python скриптов на риски и проблемы
    Переходы: self_check
    Инструменты (shared): memory, think
    Инструмен

In [3]:
print("\nВнимание: Агент будет задавать вопросы!")
print("Попробуйте ввести: 99-41116 или 99-41158\n")

result = audit_agent.invoke([
    "Проведи аудит файлов проверки. В состоянии start_work после получения списка файлов "
    "сохрани case_id, case_folder, case_files, docx_files, sql_files и py_files "
    "одним вызовом memory с параметрами keys и values (одинаковая длина списков)."
])

# Вывод результата


Внимание: Агент будет задавать вопросы!
Попробуйте ввести: 99-41116 или 99-41158

RUN 0fac4241 | AuditAgent_2055487539440 | started
STATE START -> start_work
  REQ 1 | msgs=2 in=1308 out=42
    [SYS] Ты агент аудита файлов проверки.
Твоя цель на этом шаге: найти папку проверки и собрать список файлов.

Алгоритм:
1. Попроси у пользователя номер проверки, если у тебя его нет.
2. Используй инструмент list_data_folders, чтобы увидеть доступные папки.
3. Используй find_case_folder, чтобы сопоставить ввод пользователя с папками.
4. Если статус "needs_confirmation" или есть сомнения — задай уточняющий вопрос.
5. Когда папка подтверждена, используй list_case_files и сохрани в память ВСЕ шесть ключей
   ОДНИМ вызовом memory (пакетно), без нескольких подряд вызовов memory(save) на отдельные ключи:
   memory(
     action="save",
     keys=["case_id", "case_folder", "case_files", "docx_files", "sql_files", "py_files"],
     values=[<номер проверки>, <путь к папке кейса>, <полный список файлов>, <

KeyboardInterrupt: 

---

## 🎉 Итоги тестирования

Если все тесты пройдены, значит фреймворк агентов работает корректно!

### Что было протестировано:

✅ **Базовые функции** - инструменты работают  
✅ **Test Agent** - простейший агент с 1 состоянием  
✅ **My Agent** - переходы между состояниями  
✅ **Router Agent** - условный роутинг  
✅ **Multi-Agent** - композиция агентов  
✅ **Audit Agent** - сложный workflow  

### Архитектура:

- **AgentConfig** - базовый класс для всех агентов
- **State** - декларативное описание состояний
- **Transition** - описание переходов
- **Изолированная история** - каждый агент имеет свою историю
- **Глобальная память** - инструменты memory доступны всем
- **Реестр агентов** - для мультиагентных систем

### Следующие шаги:

1. Создавайте новых агентов копированием папки и редактированием `agent.py`
2. Используйте композицию через `register_agent()` и `call_agent()`
3. Настраивайте логирование в `config.yaml` → `logging.level`: `off` / `simple` / `detailed`

In [ ]:
print("=" * 60)
print("🎉 ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!")
print("=" * 60)
print("\nФреймворк агентов готов к использованию!")
print("\nДокументация: README.md")
print("Примеры агентов: agents/*/agent.py")

🎉 ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!

Фреймворк агентов готов к использованию!

Документация: README.md
Примеры агентов: agents/*/agent.py
